# 🟠 Kafka — Parte 2: Produzindo e Consumindo Eventos em Tempo Real

---

> **Pré-requisito:** A Parte 1 deve estar com Kafka rodando (ZooKeeper + Broker na porta 9092).

> **Objetivo:** Usar a biblioteca `kafka-python` para produzir e consumir mensagens, entender offsets, consumer groups e particionamento.

| Etapa | O que faremos |
|---|---|
| **1. Setup** | Instalar kafka-python e preparar tópico |
| **2. Producer** | Enviar eventos JSON com chaves e partições |
| **3. Consumer** | Consumir eventos, controlar offsets |
| **4. Consumer Groups** | Múltiplos consumidores em paralelo |
| **5. Inspecionar** | Verificar offsets e lag pela CLI |

In [ ]:
# ============================================================
# ETAPA 1 — SETUP KAFKA (execute sempre que abrir o notebook)
# ============================================================
import subprocess, os, time, logging

logging.getLogger("kafka").setLevel(logging.CRITICAL)

KAFKA_VERSION = "3.7.0"
SCALA_VERSION = "2.13"
KAFKA_DIR     = f"/opt/kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
KAFKA_URL     = f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"
KAFKA_ARCHIVE = "/tmp/kafka.tgz"
BIN           = f"{KAFKA_DIR}/bin"

def porta_aberta(porta):
    return subprocess.run(
        ["nc", "-z", "-w1", "localhost", str(porta)],
        capture_output=True
    ).returncode == 0

def aguardar_porta(porta, label, max_seg=35):
    print(f"   Aguardando {label}", end="")
    for _ in range(max_seg):
        time.sleep(1)
        print(".", end="", flush=True)
        if porta_aberta(porta):
            print(" OK")
            return True
    print(" ERROR")
    return False

# -- 1. Java --
print("-- 1/5  Java --")
if subprocess.run(["java", "-version"], capture_output=True).returncode != 0:
    print("   Instalando Java...")
    os.system("apt-get install -y -q default-jre-headless 2>/dev/null")
print("   Java OK")

# -- 2. Netcat --
print("-- 2/5  Netcat --")
if subprocess.run(["which", "nc"], capture_output=True).returncode != 0:
    print("   Instalando netcat...")
    os.system("apt-get install -y -q netcat-openbsd 2>/dev/null")
print("   Netcat OK")

# -- 3. Kafka download + extracao --
print("-- 3/5  Kafka --")
start_sh = f"{BIN}/zookeeper-server-start.sh"

if os.path.isfile(start_sh):
    print(f"   Kafka ja instalado em {KAFKA_DIR}")
else:
    print(f"   Baixando Kafka {KAFKA_VERSION} (~113 MB)...")
    os.system(f"rm -rf {KAFKA_DIR} {KAFKA_ARCHIVE}")

    ret = os.system(f"wget -q --show-progress {KAFKA_URL} -O {KAFKA_ARCHIVE}")

    if ret != 0 or not os.path.isfile(KAFKA_ARCHIVE):
        raise RuntimeError("Download falhou. Verifique a conexao.")

    size_mb = os.path.getsize(KAFKA_ARCHIVE) / (1024 * 1024)
    print(f"   Arquivo baixado: {size_mb:.1f} MB")
    if size_mb < 50:
        raise RuntimeError(f"Arquivo muito pequeno ({size_mb:.1f} MB) — download incompleto.")

    print("   Extraindo...")
    os.system(f"tar -xzf {KAFKA_ARCHIVE} -C /opt/")

    if not os.path.isfile(start_sh):
        raise RuntimeError(f"Extracao falhou — {start_sh} nao encontrado.")

    print(f"   Kafka extraido com sucesso em {KAFKA_DIR}")

# -- 4. Configuracao do broker --
print("-- 4/5  Configuracao do Broker --")

os.system("pkill -f kafka.Kafka 2>/dev/null")
os.system("pkill -f zookeeper 2>/dev/null")
time.sleep(3)
os.system("rm -rf /tmp/kafka-logs /tmp/zookeeper")

server_props = f"{KAFKA_DIR}/config/server.properties"
os.system(f"sed -i '/^advertised\\.listeners/d' {server_props}")
os.system(f"sed -i '/^listeners=/d' {server_props}")
os.system(f"sed -i '/^log\\.dirs/d' {server_props}")
with open(server_props, "a") as f:
    f.write("\nlisteners=PLAINTEXT://0.0.0.0:9092\n")
    f.write("advertised.listeners=PLAINTEXT://localhost:9092\n")
    f.write("log.dirs=/tmp/kafka-logs\n")

check = subprocess.run(
    ["grep", "-E", "^listeners=|^advertised", server_props],
    capture_output=True, text=True
)
print(f"   {check.stdout.strip().replace(chr(10), ' | ')}")
print("   server.properties OK")

# -- 5. Servicos --
print("-- 5/5  Servicos --")

subprocess.Popen(
    f"{BIN}/zookeeper-server-start.sh {KAFKA_DIR}/config/zookeeper.properties".split(),
    stdout=open("/tmp/zookeeper.log", "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid
)
if not aguardar_porta(2181, "ZooKeeper"):
    os.system("tail -20 /tmp/zookeeper.log")
    raise RuntimeError("ZooKeeper nao subiu.")

subprocess.Popen(
    f"{BIN}/kafka-server-start.sh {KAFKA_DIR}/config/server.properties".split(),
    stdout=open("/tmp/kafka.log", "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid
)
if not aguardar_porta(9092, "Kafka Broker", max_seg=35):
    os.system("tail -20 /tmp/kafka.log")
    raise RuntimeError("Kafka Broker nao subiu.")

print("\n" + "=" * 42)
print("Kafka pronto! Pode continuar.")
print("=" * 42)

In [ ]:
!pip install kafka-python-ng --quiet

import json, threading, uuid, random
from datetime import datetime, timezone
from kafka import KafkaProducer, KafkaConsumer, KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP = "localhost:9092"
TOPIC     = "pedidos-ecommerce"

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
try:
    admin.create_topics([NewTopic(name=TOPIC, num_partitions=3, replication_factor=1)])
    print(f" Tópico '{TOPIC}' criado (3 partições)")
except TopicAlreadyExistsError:
    print(f" Tópico '{TOPIC}' já existe")

print(" Ambiente pronto — pode continuar!")

---
## 📤 Etapa 2 — Producer: Enviando Eventos

In [ ]:
# ============================================================
# ETAPA 2.1 — PRODUCER BÁSICO
# ============================================================
# O Producer serializa a mensagem para bytes antes de enviar.
# Usamos JSON como formato — padrão amplamente adotado.

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    # Serializar value como JSON bytes
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    # Serializar key como string bytes (para particionamento)
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    # Configurações de confiabilidade:
    acks='all',          # Aguardar confirmação de todas as réplicas
    retries=3,           # Tentar 3 vezes em caso de erro
    linger_ms=5,         # Agrupar mensagens por 5ms antes de enviar (batch)
)

print(" Producer criado com acks='all' e retries=3")

In [ ]:
# ============================================================
# ETAPA 2.2 — ENVIANDO EVENTOS COM KEY
# ============================================================
# A KEY determina QUAL PARTIÇÃO receberá a mensagem.
# Mensagens com a mesma key → mesma partição → ordem garantida.

CATEGORIAS = ["eletronicos", "roupas", "alimentos"]
STATUS      = ["CRIADO", "PAGO", "ENVIADO", "ENTREGUE"]

import random

def gerar_pedido(categoria):
    return {
        "pedido_id":   str(uuid.uuid4())[:8],
        "usuario_id":  f"usr_{random.randint(1, 100):03d}",
        "categoria":   categoria,
        "valor":       round(random.uniform(10, 5000), 2),
        "status":      random.choice(STATUS),
        "timestamp":   datetime.now(timezone.utc).isoformat(),
    }

print("Enviando 15 pedidos (5 por categoria)...\n")

for categoria in CATEGORIAS:
    for i in range(5):
        pedido = gerar_pedido(categoria)
        # KEY = categoria → todos os pedidos da mesma categoria
        #                    vão para a mesma partição
        future = producer.send(
            topic=TOPIC,
            key=categoria,      # Chave de particionamento
            value=pedido
        )
        metadata = future.get(timeout=5)  # Aguarda confirmação
        print(f" [{categoria:12s}] → Partição {metadata.partition} | "
              f"Offset {metadata.offset} | Pedido {pedido['pedido_id']}")

producer.flush()  # Garante envio de todas as mensagens pendentes
print(f"\n 15 mensagens enviadas com sucesso!")

In [ ]:
# ============================================================
# ETAPA 2.3 — CALLBACK DE CONFIRMAÇÃO
# ============================================================
# Em produção, usamos callbacks assíncronos para não bloquear
# o loop principal enquanto aguardamos confirmação.

enviados, erros = [], []

def on_send_success(metadata):
    enviados.append({
        'topic': metadata.topic,
        'partition': metadata.partition,
        'offset': metadata.offset
    })

def on_send_error(exc):
    erros.append(str(exc))
    print(f" Erro ao enviar: {exc}")

print("Enviando 10 pedidos com callback assíncrono...")

for i in range(10):
    cat = random.choice(CATEGORIAS)
    producer.send(
        topic=TOPIC,
        key=cat,
        value=gerar_pedido(cat)
    ).add_callback(on_send_success).add_errback(on_send_error)

producer.flush()

print(f"\n Enviados   : {len(enviados)}")
print(f"  Com erro   : {len(erros)}")
print(f"  Distribuição por partição:")

from collections import Counter
dist = Counter(m['partition'] for m in enviados)
for part, count in sorted(dist.items()):
    print(f"     Partição {part}: {count} mensagens")

---
## 📥 Etapa 3 — Consumer: Consumindo Eventos

In [ ]:

# ============================================================
# ETAPA 3.1 — CONSUMER SIMPLES
# ============================================================
# auto_offset_reset='earliest' → lê TODAS as mensagens desde o início
# auto_offset_reset='latest'   → lê apenas as NOVAS mensagens

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    group_id="grupo-analise-v1",        # Consumer Group ID
    auto_offset_reset="earliest",       # Ler do início
    enable_auto_commit=True,            # Commit automático do offset
    auto_commit_interval_ms=1000,       # Commitar a cada 1 segundo
    value_deserializer=lambda b: json.loads(b.decode('utf-8')),
    key_deserializer=lambda b: b.decode('utf-8') if b else None,
    consumer_timeout_ms=3000,           # Para de esperar após 3s sem mensagens
)

print("Consumindo mensagens...\n")
total = 0
valor_total = 0.0

for msg in consumer:
    total += 1
    valor_total += msg.value.get('valor', 0)
    print(f" Partição {msg.partition} | Offset {msg.offset:3d} | "
          f"Key: {msg.key:12s} | "
          f"Pedido: {msg.value['pedido_id']} | "
          f"R$ {msg.value['valor']:>8.2f}")

consumer.close()
print(f"\n Total consumido : {total} mensagens")
print(f"   Valor total     : R$ {valor_total:,.2f}")
print(f"   Ticket médio    : R$ {valor_total/total:,.2f}")

In [ ]:
# ============================================================
# ETAPA 3.2 — COMMIT MANUAL DE OFFSET
# ============================================================
# Commit manual = só marca como processado APÓS processar com sucesso.
# Se o processo morrer no meio, a mensagem será reprocessada.
# Isso é o padrão 'at-least-once delivery'.

consumer_manual = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    group_id="grupo-analise-v2-manual",
    auto_offset_reset="earliest",
    enable_auto_commit=False,      # ← Desabilita commit automático!
    value_deserializer=lambda b: json.loads(b.decode('utf-8')),
    key_deserializer=lambda b: b.decode('utf-8') if b else None,
    consumer_timeout_ms=3000,
    max_poll_records=5,            # Processar em batches de 5
)

print(" Consumindo com commit MANUAL (batches de 5)...\n")
batch_num = 0
total = 0

try:
    for msg in consumer_manual:
        total += 1
        print(f" Processando: {msg.value['pedido_id']} | "
              f"Status: {msg.value['status']}")

        # A cada 5 mensagens, commita o batch
        if total % 5 == 0:
            consumer_manual.commit()  # ← Commit manual!
            batch_num += 1
            print(f"  Batch {batch_num} commitado ({total} msgs processadas)\n")
finally:
    consumer_manual.commit()  # Commita o que sobrou
    consumer_manual.close()

print(f"\n {total} mensagens processadas com commit manual")
print(" Se o processo morresse antes do commit, as msgs seriam reprocessadas!")

---
## 👥 Etapa 4 — Consumer Groups

In [ ]:
# ============================================================
# ETAPA 4 — CONSUMER GROUP: MULTIPLOS CONSUMIDORES
# ============================================================
# Dentro de um grupo, cada particao e atribuida a UM consumidor.
# Com 3 particoes e 3 consumidores → processamento 3x paralelo.

# Primeiro, produzir mais 30 mensagens
print("Produzindo 30 mensagens para o teste de consumer group...")
for i in range(30):
    cat = random.choice(CATEGORIAS)
    producer.send(TOPIC, key=cat, value=gerar_pedido(cat))
producer.flush()
print("30 mensagens enviadas\n")

# Resultados por consumidor
results = {"worker-0": [], "worker-1": [], "worker-2": []}

def consumidor_worker(worker_id, group_id, results_dict, duration=8):
    """Cada worker roda em uma thread separada."""
    consumer = KafkaConsumer(
        TOPIC,
        bootstrap_servers=BOOTSTRAP,
        group_id=group_id,
        auto_offset_reset="latest",
        enable_auto_commit=True,
        value_deserializer=lambda b: json.loads(b.decode('utf-8')),
        key_deserializer=lambda b: b.decode('utf-8') if b else None,
        consumer_timeout_ms=duration * 1000,
    )
    for msg in consumer:
        results_dict[worker_id].append({
            'partition': msg.partition,
            'pedido_id': msg.value['pedido_id']
        })
    consumer.close()

GROUP_ID = "grupo-paralelo-v1"

# Subir 3 workers simultaneamente
threads = []
for wid in ["worker-0", "worker-1", "worker-2"]:
    t = threading.Thread(target=consumidor_worker,
                         args=(wid, GROUP_ID, results))
    t.start()
    threads.append(t)

# Aguardar rebalanceamento — o Kafka precisa detectar os 3 consumers,
# atribuir uma particao para cada um e confirmar o grupo antes de
# qualquer mensagem ser produzida
print("Aguardando rebalanceamento do consumer group...", end="")
time.sleep(8)
print(" OK")

# Agora sim produzir — todos os workers ja tem particao atribuida
print("Produzindo 30 mensagens com consumers ativos...")
for i in range(30):
    cat = random.choice(CATEGORIAS)
    producer.send(TOPIC, key=cat, value=gerar_pedido(cat))
producer.flush()

for t in threads:
    t.join()

print("\nDistribuicao de trabalho entre os workers:")
print(f"{'Worker':<12} {'Mensagens':>10} {'Particoes'}")
print("-" * 40)
for worker, msgs in results.items():
    partitions = set(m['partition'] for m in msgs)
    print(f"  {worker:<10} {len(msgs):>10}   {sorted(partitions)}")

print("\nCada worker processa particoes DIFERENTES — sem sobreposicao!")

---
## 🔍 Etapa 5 — Inspecionando Offsets e Lag

In [ ]:
# ============================================================
# ETAPA 5 — CONSUMER LAG
# ============================================================
# Lag = diferença entre o offset mais recente e o offset do consumer.
# Lag alto = consumer está atrasado = risco de processamento tardio.

import subprocess
KAFKA_DIR = "/opt/kafka_2.13-3.7.0"
BIN = f"{KAFKA_DIR}/bin"

print("LAG dos consumer groups:\n")

for group in ["grupo-analise-v1", "grupo-analise-v2-manual", "grupo-paralelo-v1"]:
    print(f"--- Grupo: {group} ---")
    cmd = [
        f"{BIN}/kafka-consumer-groups.sh",
        "--bootstrap-server", BOOTSTRAP,
        "--group", group,
        "--describe"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout.strip() or "(sem dados ainda)")
    print()

print("   LAG = LOG-END-OFFSET - CURRENT-OFFSET")
print("   LAG = 0 → consumer está em dia")
print("   LAG > 0 → consumer está atrasado (precisa investigar!)")